# Baseline Evaluation Notebook
Đọc `outputs/cleaned_dataset_fixed.csv` và chạy các mô hình baseline với nhiều biểu diễn văn bản khác nhau.

**Kết quả** được lưu vào `outputs/baseline_results.csv`.

**Yêu cầu:** Chạy `nlp_pipeline.ipynb` trước để có file `cleaned_dataset_fixed.csv`.

## Cell 1 – Imports

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

print("Imports OK")

Imports OK


## Cell 2 – Hàm đánh giá (`evaluate`)

In [2]:
def evaluate(name: str, X_train, X_test, y_train, y_test) -> list[dict]:
    """Huấn luyện 3 mô hình trên cùng một biểu diễn và trả về danh sách kết quả."""
    models = [
        ("NaiveBayes",         MultinomialNB()),
        ("LogisticRegression", LogisticRegression(max_iter=1200)),
        ("LinearSVM",          LinearSVC()),
    ]
    rows = []
    for model_name, model in models:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            "representation":    name,
            "model":             model_name,
            "accuracy":          float(accuracy_score(y_test, pred)),
            "precision_weighted": float(precision_score(y_test, pred, average="weighted", zero_division=0)),
            "recall_weighted":   float(recall_score   (y_test, pred, average="weighted", zero_division=0)),
            "f1_weighted":       float(f1_score        (y_test, pred, average="weighted", zero_division=0)),
        })
        print(f"  {model_name:20s} → acc={rows[-1]['accuracy']:.4f}  f1={rows[-1]['f1_weighted']:.4f}")
    return rows


print("evaluate() defined")

evaluate() defined


## Cell 3 – Đường dẫn & đọc dữ liệu

In [3]:
def find_project_root() -> Path:
    candidate = Path.cwd()
    for _ in range(4):
        if (candidate / "outputs").exists() or (candidate / "resources").exists():
            return candidate
        candidate = candidate.parent
    return Path.cwd()


project_root = find_project_root()
input_csv    = project_root / "outputs" / "cleaned_dataset_fixed.csv"
output_csv   = project_root / "outputs" / "baseline_results.csv"

print(f"project_root : {project_root}")
print(f"input_csv    : {input_csv}  (exists={input_csv.exists()})")

if not input_csv.exists():
    raise FileNotFoundError(
        f"Không tìm thấy: {input_csv}\n"
        "Hãy chạy nlp_pipeline.ipynb trước để tạo cleaned_dataset_fixed.csv."
    )

df = pd.read_csv(input_csv, encoding="utf-8-sig")

# Kiểm tra các cột cần thiết
required_cols = {"clean_text", "source"}
missing_cols  = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Dataset thiếu các cột: {missing_cols}")

# Lọc bản ghi hợp lệ
df = df.dropna(subset=["clean_text", "source"]).copy()
df["clean_text"] = df["clean_text"].astype(str)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print(f"\nSố dòng hợp lệ : {len(df):,}")
print("Phân bố nhãn:")
display(df["source"].value_counts())

project_root : c:\VIET\2-2026\NLP
input_csv    : c:\VIET\2-2026\NLP\outputs\cleaned_dataset_fixed.csv  (exists=True)

Số dòng hợp lệ : 6,905
Phân bố nhãn:


source
youtube    6787
voz         118
Name: count, dtype: int64

## Cell 4 – Chia train/test

In [4]:
X = df["clean_text"].tolist()
y = df["source"].astype(str).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test  size : {len(X_test):,}")

Train size : 5,524
Test  size : 1,381


## Cell 5 – One-Hot (Binary CountVectorizer)

In [5]:
print("=== One-Hot (Binary CountVectorizer) ===")
rows = []

one_hot = CountVectorizer(max_features=3000, binary=True)
rows.extend(evaluate(
    "OneHot",
    one_hot.fit_transform(X_train),
    one_hot.transform(X_test),
    y_train, y_test,
))

=== One-Hot (Binary CountVectorizer) ===
  NaiveBayes           → acc=0.9884  f1=0.9873
  LogisticRegression   → acc=0.9884  f1=0.9856
  LinearSVM            → acc=0.9906  f1=0.9889


## Cell 6 – CountVectorizer (Bag of Words)

In [6]:
print("=== CountVectorizer (BoW) ===")
count = CountVectorizer(max_features=6000)
rows.extend(evaluate(
    "CountVectorizer",
    count.fit_transform(X_train),
    count.transform(X_test),
    y_train, y_test,
))

=== CountVectorizer (BoW) ===
  NaiveBayes           → acc=0.9884  f1=0.9856
  LogisticRegression   → acc=0.9884  f1=0.9856
  LinearSVM            → acc=0.9884  f1=0.9861


## Cell 7 – N-grams (1-2)

In [7]:
print("=== N-grams (1,2) ===")
ngrams = CountVectorizer(max_features=12000, ngram_range=(1, 2))
rows.extend(evaluate(
    "Ngrams_1_2",
    ngrams.fit_transform(X_train),
    ngrams.transform(X_test),
    y_train, y_test,
))

=== N-grams (1,2) ===
  NaiveBayes           → acc=0.9884  f1=0.9856
  LogisticRegression   → acc=0.9884  f1=0.9856
  LinearSVM            → acc=0.9891  f1=0.9867


## Cell 8 – Hashing Vectorizer
> `HashingVectorizer` không cần `fit` nên dùng `transform()` trực tiếp cho cả train và test.

In [8]:
print("=== HashingVectorizer ===")
# Lưu ý: MultinomialNB yêu cầu giá trị không âm → dùng alternate_sign=False
hashing = HashingVectorizer(n_features=2**14, alternate_sign=False, norm=None)
rows.extend(evaluate(
    "Hashing",
    hashing.transform(X_train),  # HashingVectorizer không có fit
    hashing.transform(X_test),
    y_train, y_test,
))

=== HashingVectorizer ===
  NaiveBayes           → acc=0.9826  f1=0.9775
  LogisticRegression   → acc=0.9884  f1=0.9856
  LinearSVM            → acc=0.9884  f1=0.9861


## Cell 9 – TF-IDF (1-2 grams)

In [9]:
print("=== TF-IDF (1,2) ===")
tfidf = TfidfVectorizer(max_features=12000, ngram_range=(1, 2))
rows.extend(evaluate(
    "TFIDF",
    tfidf.fit_transform(X_train),
    tfidf.transform(X_test),
    y_train, y_test,
))

=== TF-IDF (1,2) ===
  NaiveBayes           → acc=0.9826  f1=0.9740
  LogisticRegression   → acc=0.9841  f1=0.9774
  LinearSVM            → acc=0.9899  f1=0.9878


## Cell 10 – Tổng hợp kết quả & lưu file

In [10]:
result = (
    pd.DataFrame(rows)
    .sort_values(by=["f1_weighted", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

output_csv.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu: {output_csv}")
display(result)

✅ Đã lưu: c:\VIET\2-2026\NLP\outputs\baseline_results.csv


,representation,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,OneHot,LinearSVM,0.990587,0.990676,0.990587,0.988861
1,TFIDF,LinearSVM,0.989862,0.989966,0.989862,0.987801
2,OneHot,NaiveBayes,0.988414,0.987017,0.988414,0.987273
3,Ngrams_1_2,LinearSVM,0.989138,0.989257,0.989138,0.986700
4,CountVectorizer,LinearSVM,0.988414,0.987511,0.988414,0.986059
5,Hashing,LinearSVM,0.988414,0.987511,0.988414,0.986059
6,OneHot,LogisticRegression,0.988414,0.988549,0.988414,0.985552
7,CountVectorizer,NaiveBayes,0.988414,0.988549,0.988414,0.985552
8,CountVectorizer,LogisticRegression,0.988414,0.988549,0.988414,0.985552
9,Ngrams_1_2,NaiveBayes,0.988414,0.988549,0.988414,0.985552
